# 2-motor: bemf-coast

Measure the motor's back-EMF constant from the terminal voltage of the coasting rotor over its speed, with the speed counted in the commutation ripple of the same samples, so every speed read from the terminals later rests on a number taken with no current in the winding.

## Purpose

A brushed DC motor turning at a speed $\omega$ puts a voltage $e = K_e \, \omega$ across its terminals, the back-EMF, and $K_e$ is the back-EMF constant in volt seconds per radian. The same constant is the torque per amp, $K_t = K_e$ in SI units, so it ties the current to the torque as well. Stage 3 reads the motor's speed from the terminals in situ through $K_e$, stage 4 builds the speed estimator on it, and the no-load-current experiment of this stage turns the spin's current into a drag torque through it. A motor has no datasheet, so the number has to be measured, and the cleanest place to measure it is the bare motor with nothing between the terminals and the board.

The route is the open-circuit one. The motor is spun at a duty, then the bridge is opened and the rotor coasts. With both bridge legs off the winding carries no current, so the terminal voltage is the EMF itself with no resistive drop to take out, and the EMF carries the commutation ripple, $N_r$ events per revolution, which counts the speed in the same samples. Nothing outside the board is needed. A second route reads the EMF at the cut over the speed the spin's current ripple counts, and the two are compared cycle by cycle. The coast route is the one the result file carries. Over the duties and the two rail levels the cycles span a range of speeds, and whether one $K_e$ holds across that range, across the directions and across the levels is part of what gets settled.

## Procedure

`capture.py` runs cycles of a spin and a coast. A cycle is one `osc sweep` with the schedule `D@1500,coast:800` in both directions. It records a half second torque-off rest, then 1.5 s at the duty $D$ forward, then 0.8 s with the bridge open and the rotor coasting, then the same pair in reverse. The coast is chained to its spin with no settle between them, so the rotor goes from the drive into the coast at speed. The spin is long enough for the speed to settle, which an earlier run with a 300 ms spin did not give, and the Method shows what the settle looks like in the current's tail. The duties are 15, 20, 30, 40 and 60 %, three cycles each, at rail levels of 5.0 and 7.4 V, thirty cycles in all, with twenty seconds of rest between cycles, the bench rule for a bare motor.

1. The bare motor in the rig, its two leads on the board's motor connector, the wrench bar off the pinion so the rotor is free. The board on the DPS, the adapter on the bus at 3 Mbps. The board runs the experiment image, so the cycles are open loop with only the raw current trip and the bus dead-man as protections.
2. `capture.py`. The DPS comes on before the board's leads go on, because its output comes up at its previous setting for a moment. The script then trims the DPS until the board's own rail tap reads each level, and runs the fifteen cycles of that level.

The levels name the rail, VSYS. The dev board has a feed diode, so the DPS is trimmed over it until the rail reads the level, and the trim reads the rail through the chip's own 1.2 V reference.

## Data

One folder under the capture.

- `cycles.csv`: one row per cycle. `k` the cycle number, `volts_set` the rail level asked for, `dps_read` the DPS output that reached it, `vsys_read` the rail the board read at rest before the level's cycles, `duty_pct` the spin duty, `repeat` 0 to 2, `t_start` and `t_end` the host time the sweep ran, `folder` the cycle's folder.
- `cycles/K/sweep.csv`, `meta.json`: the rest, the two spins and the two coasts of one cycle, one row per control tick at 20 kHz. Segment 0 is the rest, 1 the forward spin, 2 the coast chained to it, 3 and 4 the same in reverse. `cmd_duty_q15` is the commanded duty, 0 in a coast, `dir` the direction, `duty_counts` the applied window in timer ticks, 0 from the cut on. `current_raw`, `vmotor_a`, `vmotor_b`, `vbus_raw` and `ntc_raw` are ADC counts. `tick` restarts at 0 in every segment, and a jump in it is a dropped frame. Each segment is its own stream from the board, and the rows between the end of a spin's stream and the start of its coast's stream, while the host commits the zero duty, are not recorded.

### cycles.csv

The rail asked for and the duty of every cycle, in the order they ran.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import butter, sosfiltfilt

from oscnb import boards, motors, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"   # the house colors
MOTOR = motors.MOTORS[motors.DEFAULT]                                   # the motor, with its counted parts
BOARD = boards.BOARDS[boards.DEFAULT]                                   # the board, with the parts' nominal values
CAPTURE = (Path("../../../telemetry") / f"{MOTOR.key}__{BOARD.key}" / "bemf-coast" / "capture-1").resolve()
COLOR = {5.0: BLUE, 7.4: ORANGE}                                        # one color per rail level, fixed through the notebook
MARK = {1: "o", -1: "^"}                                                # the direction as the marker shape, forward filled, reverse open
FS = BOARD.tick_hz                                                      # rows per second, the control rate
N_R = MOTOR.ripple_per_rev                                              # commutation events per revolution, slots x brushes
print(MOTOR.key, "on", BOARD.key, "from", results.rel(CAPTURE))
print(f"{MOTOR.slots} slots x {MOTOR.brushes} brushes = {N_R} ripple events per revolution; rows at {FS} Hz")

In [ ]:
cycles = pd.read_csv(CAPTURE / "cycles.csv")
cycles["trim_mv"] = (cycles["vsys_read"] - cycles["volts_set"]) * 1e3   # how far the trim landed from the level
show(cycles, n=None)
fig, ax = plt.subplots(figsize=(9, 3))
for v, g in cycles.groupby("volts_set"):
    ax.plot(g["k"], g["duty_pct"], "o", color=COLOR[v], label=f"{v:g} V, rail read {g['vsys_read'].iloc[0]:.3f} V")   # one dot per cycle
ax.set_xlabel("cycle k"); ax.set_ylabel("spin duty (%)"); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"{len(cycles)} cycles, {cycles['t_end'].iloc[-1] - cycles['t_start'].iloc[0]:.0f} s from the first sweep to the end of the last")

The duty ladder at each level, three cycles per duty, with the rail the board read before each level's cycles in the legend. The thing to see is that the levels landed where they were asked, within a few tens of millivolts.

### cycles/K/sweep.csv and meta.json

One cycle, the first 30 % cycle at 5.0 V. The current channel, the two terminal taps and the applied duty over the rest, the two spins and the two coasts, then a close look at the cut.

In [ ]:
def cycle_folder(k):
    return CAPTURE / cycles.loc[cycles["k"] == k, "folder"].iloc[0]

def pairs(df):
    # (direction, spin rows, coast rows) for every spin and the coast chained to it: a coast is a
    # driven segment with a zero command, and its spin is the segment before it
    out = []
    for seg, co in df[df["seg"] > 0].groupby("seg"):
        if int(co["cmd_duty_q15"].iloc[0]) != 0:
            continue
        sp = df[df["seg"] == seg - 1]
        out.append((int(co["dir"].iloc[0]), sp, co))
    return out

def cut_row(co):
    # the cut: the first row of the coast's stream with no drive, the rows before it being the drive's tail
    z = np.flatnonzero(co["duty_counts"].to_numpy() == 0)
    return int(z[0]) if len(z) else None

K_SHOW = int(cycles[(cycles["volts_set"] == 5.0) & (cycles["duty_pct"] == 30)]["k"].iloc[0])
df, meta = tel.load(cycle_folder(K_SHOW), BOARD)      # checks the sense block against the board entry
print("cycle", K_SHOW, "| segments:", df["seg"].value_counts().sort_index().to_dict(), "| schedule:", meta["schedule"],
      "| dirs:", meta["dirs"], "| baseline ms:", meta["baseline_ms"], "| decay:", meta["decay"], "| rows dropped:", meta["rows_dropped"])
SPIN_LEN_MS = len(pairs(df)[0][1]) / FS * 1e3          # the spin's length in ms, from its rows
ts = np.arange(len(df)) / FS                           # time through the sweep as rows, since `tick` restarts at each segment
edges = np.flatnonzero(np.diff(df["seg"].to_numpy()) != 0) + 1
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(ts, df["current_raw"], lw=0.4, color=BLUE); ax[0].set_ylabel("current_raw (counts)")
ax[1].plot(ts, df["vmotor_a"], lw=0.4, color=BLUE, label="tap A"); ax[1].plot(ts, df["vmotor_b"], lw=0.4, color=ORANGE, label="tap B")
ax[1].set_ylabel("terminal taps (counts)"); ax[1].legend(loc="upper right")
ax[2].plot(ts, df["duty_counts"], lw=0.8, color=INK); ax[2].set_ylabel("duty_counts (ticks)"); ax[2].set_xlabel("time through the sweep, rows only (s)")
for a in ax:
    for e in edges:
        a.axvline(e / FS, color=MUTED, lw=0.6, ls=":")   # segment edges, where the streams meet
    a.grid(alpha=0.3)
ax[0].set_title(f"cycle {K_SHOW}: the rest, then a spin and its coast each way, segment edges dotted")
plt.tight_layout(); plt.show()

The current steps up with the spin and falls to the rest zero in the coast, the taps swap with the direction and decay together through each coast, and the duty shows the two spins and the two coasts as four flat stretches. The rows between the segments are not recorded, so the streams sit end to end here.

In [ ]:
d, sp, co = pairs(df)[0]
z = cut_row(co)
tail, head = sp.iloc[-400:], co.iloc[:z + 400]        # the last 20 ms of the spin's stream, the first 20 ms past the cut
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(np.arange(-len(tail), 0) / FS * 1e3, (tail["vmotor_a"] - tail["vmotor_b"]) * d, lw=0.8, color=MUTED, label="spin, the last 20 ms of its stream")
ax[0].plot(np.arange(len(head)) / FS * 1e3, (head["vmotor_a"] - head["vmotor_b"]) * d, lw=0.8, color=BLUE, label="coast stream")
ax[0].axvline(z / FS * 1e3, color=INK, lw=0.8, ls="--", label="the cut, duty_counts first 0")
ax[0].set_ylabel("tap A - tap B, signed by direction (counts)"); ax[0].legend(fontsize=8); ax[0].set_title("the terminal difference across the cut")
ax[1].plot(np.arange(-len(tail), 0) / FS * 1e3, tail["current_raw"], lw=0.8, color=MUTED); ax[1].plot(np.arange(len(head)) / FS * 1e3, head["current_raw"], lw=0.8, color=BLUE)
ax[1].axvline(z / FS * 1e3, color=INK, lw=0.8, ls="--"); ax[1].set_ylabel("current_raw (counts)"); ax[1].set_title("the current across the cut")
for a in ax:
    a.set_xlabel("ms from the start of the coast's stream, the spin's tail placed before it"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"direction {d:+d}: the coast's stream opens with {z} rows of drive before the cut; the coast holds {len(co) - z} rows past it, {(len(co) - z) / FS:.3f} s")

Under drive the difference reads the rail at the crest sample and the current carries the ripple. From the cut the difference drops to the EMF and starts to decay, with the same ripple on it, and the current falls to the rest zero. The gap between the two streams is not drawn, the spin's tail is just placed before the coast's stream.

## Raw traces

Every coast's terminal difference from its cut, one panel per level, light to dark with the spin duty, so the decays can be compared across duties and the repeats seen on top of each other.

In [ ]:
DUTIES = sorted(cycles["duty_pct"].unique())
SHADE = {v: dict(zip(DUTIES, (plt.cm.Blues if v == 5.0 else plt.cm.Oranges)(np.linspace(0.35, 0.95, len(DUTIES))))) for v in COLOR}   # one hue per level, light to dark with the duty
missing = {}
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for _, C in cycles.iterrows():
    df, meta = tel.load(cycle_folder(C["k"]), BOARD)
    missing[int(C["k"])] = int(tel.gaps(df).sum())
    a = ax[0] if C["volts_set"] == 5.0 else ax[1]
    for d, sp, co in pairs(df):
        z = cut_row(co)
        if z is None:
            continue
        e = ((co["vmotor_a"] - co["vmotor_b"]) * d).to_numpy(float)[z:]
        a.plot(np.arange(len(e)) / FS, e, lw=0.4, color=SHADE[C["volts_set"]][C["duty_pct"]], label=f"{C['duty_pct']} %" if C["repeat"] == 0 and d == 1 else None)
for a, v in zip(ax, COLOR):
    a.set_title(f"{v:g} V: every coast's terminal difference, light to dark with the spin duty"); a.set_xlabel("time from the cut (s)"); a.grid(alpha=0.3); a.legend(fontsize=8, title="duty")
ax[0].set_ylabel("tap A - tap B, signed by direction (counts)")
plt.tight_layout(); plt.show()
print("missing rows per cycle:", {k: m for k, m in missing.items() if m} or "none")

Each coast starts at the EMF its spin reached and decays toward zero, faster from the higher duties if the drag rises with speed. The three repeats of a duty should lie on top of each other, and a cycle whose rotor did not turn shows as a flat line at zero.

## Method

Two symbols carry the notebook. $\omega$ is the motor's speed in radians per second, and $K_e$ the back-EMF constant, so that the open-circuit terminal voltage is $e = K_e \, \omega$. The speed is never read from an outside instrument here. It is counted in the ripple. A brushed motor with $N_r$ commutation events per revolution, 3 slots times 2 brushes for this one, puts a ripple at $f_r = N_r \, \omega / 2\pi$ hertz on its current under drive and on its terminal voltage when it coasts, so a count of the ripple's cycles over a known time is a speed, with the parts count as the only constant.

Every cycle goes through the same reduction, once per direction.

1. The rest. The second half of the torque-off segment gives the current channel's zero and the taps' difference at rest, with no EMF and no current. Both are the cycle's own.
2. The spin window, the last 200 ms of the spin's stream. The current over the rest zero and the stage 1 in-drive offset, converted through the current-scale result. The applied duty, the rail and the driven terminal level. The spin's speed, counted in the current's ripple over the window.
3. The coast, from the cut, the first row of the coast's stream whose applied duty is zero. The terminal difference signed by the direction, less the rest difference, converted through the terminal-chain result, is the EMF $e(t)$. A cycle turned if the EMF 0.1 to 1 ms past the cut is at least 0.05 V. A rotor that did not turn leaves nothing to count.
4. The coast's ripple line, the strongest line in the spectrum of the EMF 1 to 11 ms past the cut. It seeds both counts. $E_0$, the EMF at the cut, is a line through the one-ripple boxcar of the EMF over those 10 ms extrapolated back to the cut, and its slope is the deceleration the no-load-current experiment reads.
5. The coast route. The EMF over a whole number of ripple cycles inside the same 10 ms, found as the upward zero crossings of the band-passed EMF, over the speed those crossings count,

$$K_e = \frac{\overline{e}}{\omega} = \frac{\overline{e} \, N_r}{2\pi \, f_r}$$

with the mean and the count taken over the same rows. No current flows, so there is no $R \, i$ to take out and no resistance to know. Whole cycles keep the ripple out of the mean.

6. The spin route. $E_0$ over the spin window's counted speed. This assumes the speed does not move between the window and the cut, which the gap between the streams does not let the data check directly, so the paired difference of the two routes is the check.

The spin's count is seeded by the coast's line and band-passed around it. Under drive the current can carry a line at half the ripple frequency, three events per revolution, that is stronger than the six-per-revolution line at the higher duties, and a free peak search would land on it and halve the speed. The seed keeps the count on the same line the coast reads, and the height of the half line over the full line in the spin's spectrum is kept in the table.

The 10 ms window is a trade. The EMF decays through it, so a longer window averages over more of the decay and a shorter one holds fewer ripple cycles to count. The ratio of the window's mean EMF to its mean frequency is what the method reads, and the decay enters both. With the coast 0.8 s long the same window can be stepped down the whole coast, which reads $K_e$ at every speed the rotor passes through and shows whether it holds at the low end, where the slowest duties sit.

The spin is 1.5 s so the speed has settled when the window is read. The settle is shown rather than assumed. The spin's current in 50 ms bins, with the speed counted in the same bins, says how the rotor approached its steady speed, and an exponential fitted to the tail from 300 ms on gives the time constant of the approach and how far the window's mean sits from the asymptote. The no-load-current experiment reads the same window for the drag, so the settle is its margin as much as this experiment's.

### Stage 1 values carried in

In [ ]:
TC = results.read("1-board", "terminal-chain", pair=BOARD.key)     # the taps' and the rail's counts per volt against the meter
CS = results.read("1-board", "current-scale", pair=BOARD.key)      # the current chain's counts per amp and its in-drive offset
RN = results.read("1-board", "rest-noise", pair=BOARD.key)         # the taps' floor at rest
V_PER_COUNT = 1 / TC["terminal_counts_per_v"]["value"]              # volts of terminal difference per count
A_PER_COUNT = 1 / CS["counts_per_a"]["value"]                        # amps per count over the zero
I_DRIVE_OFFSET = CS["drive_offset_counts"]["value"]                 # counts the chain reads over its rest zero whenever the bridge is active
rail_v = lambda n: (np.asarray(n, float) - TC["rail_offset_counts"]["value"]) / TC["rail_counts_per_v"]["value"]
tap_floor = np.hypot(RN["tap_a_noise"]["value"], RN["tap_b_noise"]["value"])   # one row of difference, both taps' floors added in quadrature
print(f"terminal difference {TC['terminal_counts_per_v']['value']:.2f} counts/V ({TC['terminal_counts_per_v']['lo']:.2f} to {TC['terminal_counts_per_v']['hi']:.2f}), "
      f"{V_PER_COUNT * 1e3:.3f} mV per count, {(TC['terminal_counts_per_v']['value'] * BOARD.v_term_per_count - 1) * 100:+.2f} % over the nominal")
print(f"current {CS['counts_per_a']['value']:.1f} counts/A ({CS['counts_per_a']['lo']:.1f} to {CS['counts_per_a']['hi']:.1f}), in-drive offset {I_DRIVE_OFFSET:+.1f} counts, "
      f"graded on DC over {CS['amps_range']['lo']:.2f} to {CS['amps_range']['hi']:.2f} A")
print(f"rail {TC['rail_counts_per_v']['value']:.2f} counts/V at rest, offset {TC['rail_offset_counts']['value']:.1f} counts")
print(f"taps at rest {RN['tap_a_noise']['value']:.2f} and {RN['tap_b_noise']['value']:.2f} counts rms, so one row of difference carries about {tap_floor * V_PER_COUNT * 1e3:.1f} mV of noise")

### The per-cycle reduction

Every cycle through the steps above, one row per direction.

In [ ]:
SPIN_MS = 200                      # the spin window: the last 200 ms of the spin's stream, where the speed has settled
COAST_FROM, COAST_TO = 20, 220     # rows past the cut the EMF is read over, 1 to 11 ms
TURN_V = 0.05                      # a coast whose EMF 0.1 to 1 ms past the cut is under this did not turn
BAND = (150.0, 6000.0)             # Hz, where a ripple line is looked for

def spectrum(x):
    # the Hann-windowed amplitude spectrum, zero-padded 8x, and its frequencies
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    return np.fft.rfftfreq(8 * n, 1 / FS), np.abs(np.fft.rfft(x * np.hanning(n), 8 * n))

def line_freq(x, lo=BAND[0], hi=BAND[1]):
    # (frequency, prominence) of the strongest line in [lo, hi] Hz, the prominence its height over the band's median
    f, X = spectrum(x)
    m = (f >= lo) & (f <= hi)
    k = int(np.argmax(X[m]))
    return float(f[m][k]), float(X[m][k] / np.median(X[m]))

def line_height(x, f0, frac=0.2):
    # the tallest point of the spectrum within f0 x (1 +/- frac)
    f, X = spectrum(x)
    m = (f >= f0 * (1 - frac)) & (f <= f0 * (1 + frac))
    return float(X[m].max())

def count_cycles(x, f0, frac=0.35):
    # (mean frequency, first row, last row, cycles) over a whole number of cycles of the line near f0: a
    # second-order band-pass f0 x (1 +/- frac) run forward and back, upward zero crossings interpolated to a
    # fraction of a row, the first and last crossing dropped as filter edge; NaN and zero cycles under four crossings
    hi = min(f0 * (1 + frac), 0.95 * FS / 2)
    sos = butter(2, [f0 * (1 - frac), hi], btype="band", fs=FS, output="sos")
    y = sosfiltfilt(sos, np.asarray(x, float) - np.mean(x))
    s = np.flatnonzero((y[:-1] < 0) & (y[1:] >= 0))
    if len(s) < 4:
        return np.nan, 0, 0, 0
    tc = s + (-y[s]) / (y[s + 1] - y[s])
    tc = tc[1:-1]
    n = len(tc) - 1
    return n / ((tc[-1] - tc[0]) / FS), int(np.ceil(tc[0])), int(np.floor(tc[-1])), n

def reduce_cycle(C):
    # one cycle through the reduction, one row per direction
    df, meta = tel.load(cycle_folder(C["k"]), BOARD)
    rest = tel.baseline(df)
    zero_i = rest["current_raw"].median()                             # the current channel's own rest zero
    zero_e = (rest["vmotor_a"] - rest["vmotor_b"]).median()            # the tap difference at rest, no EMF and no current
    rest_tap = {"vmotor_a": rest["vmotor_a"].median(), "vmotor_b": rest["vmotor_b"].median()}
    rows = []
    for d, sp, co in pairs(df):
        row = {"k": int(C["k"]), "level": float(C["volts_set"]), "duty": int(C["duty_pct"]), "repeat": int(C["repeat"]), "dir": d,
               "missing": int(tel.gaps(df).sum())}
        # the spin window: current over the rest zero and the in-drive offset, the applied duty, the rail, the driven terminal level
        w = sp.iloc[-SPIN_MS * FS // 1000:]
        row["i_spin_a"] = (w["current_raw"].mean() - zero_i - I_DRIVE_OFFSET) * A_PER_COUNT
        row["i_spin_sd_a"] = w["current_raw"].std() * A_PER_COUNT
        row["duty_applied"] = float(BOARD.duty_pct_of_ticks(w["duty_counts"].abs().median()))
        row["rail_v"] = float(rail_v(w["vbus_raw"].mean()))
        row["v_on"] = ((w["vmotor_a"] - w["vmotor_b"]) * d - zero_e).mean() * V_PER_COUNT
        # the coast, from the cut
        z = cut_row(co)
        if z is None:
            rows.append(row); continue
        e = (((co["vmotor_a"] - co["vmotor_b"]) * d).to_numpy(float)[z:] - zero_e) * V_PER_COUNT
        row["e_first_v"] = e[2:20].mean()
        row["turned"] = bool(row["e_first_v"] >= TURN_V)
        if not row["turned"]:
            rows.append(row); continue
        t = np.arange(len(e)) / FS
        f_c0, prom = line_freq(e[COAST_FROM:COAST_TO])                  # the early coast's ripple line, the seed of both counts
        nbox = max(int(round(FS / f_c0)), 1)
        es = np.convolve(e, np.ones(nbox) / nbox, mode="same")          # one-ripple boxcar
        p = np.polyfit(t[COAST_FROM:COAST_TO], es[COAST_FROM:COAST_TO], 1)
        row.update(e0_v=float(np.polyval(p, 0.0)), dedt_v_per_s=float(p[0]), f_seed=f_c0, seed_prominence=prom)
        f_c, b0, b1, n_c = count_cycles(e[COAST_FROM:COAST_TO], f_c0)   # the coast route: whole cycles, the EMF over the same rows
        row.update(f_coast=f_c, coast_cycles=n_c, emf_v=e[COAST_FROM + b0:COAST_FROM + b1].mean() if n_c else np.nan)
        # the two taps over the counted rows, each from its own rest, the positive terminal first
        pos, neg = ("vmotor_a", "vmotor_b") if d == 1 else ("vmotor_b", "vmotor_a")
        win = co.iloc[z + COAST_FROM + b0:z + COAST_FROM + b1]
        if n_c:
            row["tap_pos_v"] = (win[pos].mean() - rest_tap[pos]) * V_PER_COUNT
            row["tap_neg_v"] = (win[neg].mean() - rest_tap[neg]) * V_PER_COUNT
        # the spin's speed, counted in the current's ripple over the window, seeded by the coast's line
        x = w["current_raw"].to_numpy(float)
        f_s, _, _, n_s = count_cycles(x, f_c0)
        row.update(f_spin=f_s, spin_cycles=n_s, spin_half_over_full=line_height(x, f_c0 / 2) / line_height(x, f_c0))
        rows.append(row)
    return rows

CYC = pd.DataFrame([r for _, C in cycles.iterrows() for r in reduce_cycle(C)])
CYC["turned"] = CYC["turned"].fillna(False).astype(bool)
CYC["speed_rev_s"] = CYC["f_coast"] / N_R
CYC["w_coast"] = 2 * np.pi * CYC["f_coast"] / N_R              # rad/s at the motor shaft, from the coast's count
CYC["w_spin"] = 2 * np.pi * CYC["f_spin"] / N_R                # the same from the spin's current
CYC["ke_coast"] = CYC["emf_v"] / CYC["w_coast"]                # V s/rad, no current in the winding
CYC["ke_spin"] = CYC["e0_v"] / CYC["w_spin"]                   # the EMF at the cut over the drive's speed
show(CYC[["k", "level", "duty", "dir", "duty_applied", "rail_v", "i_spin_a", "v_on", "e_first_v", "turned", "e0_v", "f_spin", "spin_half_over_full",
          "f_coast", "coast_cycles", "speed_rev_s", "emf_v", "ke_coast", "ke_spin"]], n=None)
print(f"{len(CYC)} spin and coast pairs, {int(CYC['turned'].sum())} turned, {int(CYC['ke_coast'].notna().sum())} with a counted coast; "
      f"missing rows in {int((CYC['missing'] > 0).sum())} cycles")

One row per spin and coast pair. A cycle that did not turn has no coast count and no $K_e$. The half-over-full column is the spin current's line at three per revolution over its line at six, and a value over one says the free peak would have landed on the wrong line.

### The settle

The spin's current in 50 ms bins and the speed counted in the same bins, averaged over each duty's cycles, with the window shaded, then the tail fit per spin.

In [ ]:
BIN_MS = 50
rows = []
for _, C in cycles.iterrows():
    df, meta = tel.load(cycle_folder(C["k"]), BOARD)
    zero_i = tel.baseline(df)["current_raw"].median()
    for d, sp, co in pairs(df):
        x = sp["current_raw"].to_numpy(float)
        seed = CYC[(CYC["k"] == C["k"]) & (CYC["dir"] == d)]["f_seed"] if "f_seed" in CYC else pd.Series(dtype=float)
        f0 = float(seed.iloc[0]) if len(seed) and np.isfinite(seed.iloc[0]) else np.nan
        nb = BIN_MS * FS // 1000
        for b in range(len(x) // nb):
            seg = x[b * nb:(b + 1) * nb]
            wide = x[max(b * nb - nb // 2, 0):(b + 1) * nb + nb // 2]                     # half a bin of margin each side for the filter
            f = count_cycles(wide, f0, frac=0.5)[0] if np.isfinite(f0) and b >= 2 else np.nan   # a wider band: the speed is still climbing early on
            rows.append({"k": int(C["k"]), "level": float(C["volts_set"]), "duty": int(C["duty_pct"]), "dir": d, "t_ms": BIN_MS * b + BIN_MS / 2,
                         "i_ma": (seg.mean() - zero_i - I_DRIVE_OFFSET) * A_PER_COUNT * 1e3, "rev_s": f / N_R})
SET = pd.DataFrame(rows)
fig, ax = plt.subplots(2, 2, figsize=(14, 7.5), sharex=True)
for j, v in enumerate(COLOR):
    for duty, g in SET[SET["level"] == v].groupby("duty"):
        m = g.groupby("t_ms").mean(numeric_only=True)
        ax[0, j].plot(m.index, m["i_ma"], "-", lw=1.2, color=SHADE[v][duty], label=f"{duty} %")
        ax[1, j].plot(m.index, m["rev_s"], "-", lw=1.2, color=SHADE[v][duty], label=f"{duty} %")
    ax[0, j].set_title(f"{v:g} V: the spin's current in {BIN_MS} ms bins, mean of the duty's cycles"); ax[1, j].set_title(f"{v:g} V: the speed counted in the same bins")
    ax[1, j].set_xlabel("ms into the spin")
    for a in ax[:, j]:
        a.axvspan(SPIN_LEN_MS - SPIN_MS, SPIN_LEN_MS, color=MUTED, alpha=0.15); a.grid(alpha=0.3); a.legend(fontsize=8, title="duty")
ax[0, 0].set_ylabel("current (mA)"); ax[1, 0].set_ylabel("speed (rev/s)")
plt.tight_layout(); plt.show()

# the tail of each spin: an exponential approach fitted from 300 ms on, its asymptote against the window's mean
tails = []
for (k, d), g in SET.groupby(["k", "dir"]):
    g = g[g["t_ms"] >= 300]
    tt, ii, ss = g["t_ms"].to_numpy(float), g["i_ma"].to_numpy(float), g["rev_s"].to_numpy(float)
    last = SPIN_MS // BIN_MS                                                  # the bins inside the window
    try:
        p, _ = curve_fit(lambda x, a, A, tau: a + A * np.exp(-(x - 300) / tau), tt, ii, p0=[ii[-last:].mean(), 5.0, 200.0],
                         bounds=([0, -500, 10], [1000, 500, 5000]))
    except (RuntimeError, ValueError):
        continue
    tails.append({"k": k, "dir": d, "i_inf_ma": p[0], "tail_tau_ms": p[2], "window_less_inf_ma": ii[-last:].mean() - p[0],
                  "speed_drift_pct": (ss[-1] / ss[-last] - 1) * 100 if np.isfinite(ss[-last:]).all() else np.nan})
TAIL = pd.DataFrame(tails)
CYC = CYC.merge(TAIL, on=["k", "dir"], how="left")
show(TAIL.describe().loc[["mean", "std", "min", "max"]], n=None)
print(f"tail time constant median {TAIL['tail_tau_ms'].median():.0f} ms ({TAIL['tail_tau_ms'].min():.0f} to {TAIL['tail_tau_ms'].max():.0f}); "
      f"the window's mean sits {TAIL['window_less_inf_ma'].mean():+.2f} mA (sd {TAIL['window_less_inf_ma'].std():.2f}) over the asymptote; "
      f"the speed moves {TAIL['speed_drift_pct'].mean():+.2f} % (sd {TAIL['speed_drift_pct'].std():.2f}) across the window")

The current should fall and the speed climb through the first few hundred milliseconds and both go flat long before the window. The tail table says how flat, with the time constant of the approach, how far the window's mean current sits from where the tail is heading, and how much the counted speed moves across the window. A window within a milliamp and a fraction of a percent of speed is settled for both this experiment and no-load-current.

## Results

### Ke per cycle, both routes

Every cycle's $K_e$ by the coast route and by the spin route against the spin duty, one panel per level, the direction as the marker, with the cycles' mean dotted. The thing to see is whether the coast route sits flat across the duties and the two routes sit on each other.

In [ ]:
ok = CYC[CYC["turned"] & CYC["ke_coast"].notna()]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for a, v in zip(ax, COLOR):
    g = ok[ok["level"] == v]
    for d in (1, -1):
        h = g[g["dir"] == d]
        a.plot(h["duty"] - 0.6, h["ke_coast"] * 1e3, MARK[d], color=COLOR[v], mfc=COLOR[v] if d == 1 else "none", label=f"coast route, direction {d:+d}")
        a.plot(h["duty"] + 0.6, h["ke_spin"] * 1e3, MARK[d], color=MUTED, mfc=MUTED if d == 1 else "none", label=f"spin route, direction {d:+d}")
    a.axhline(ok["ke_coast"].mean() * 1e3, color=INK, lw=0.8, ls=":")
    a.set_title(f"{v:g} V: $K_e$ per cycle, both routes, offset in duty to separate them"); a.set_xlabel("spin duty (%)"); a.grid(alpha=0.3); a.legend(fontsize=8)
ax[0].set_ylabel("$K_e$ (mV s/rad)")
plt.tight_layout(); plt.show()

def stats(x):
    x = x.dropna()
    return {"n": len(x), "mean_v_s_per_rad": x.mean(), "sd_pct": x.std() / x.mean() * 100, "se_pct": x.std() / np.sqrt(len(x)) / x.mean() * 100}
show(pd.DataFrame({"coast route": stats(ok["ke_coast"]), "spin route": stats(ok["ke_spin"])}).T, n=None)
by = ok.groupby(["level", "duty"])["ke_coast"].agg(["count", "mean", "std"])
by["sd_pct"] = by["std"] / by["mean"] * 100
show(by, n=None)
KE_MEAN, KE_SD = ok["ke_coast"].mean(), ok["ke_coast"].std()
print(f"Ke by the coast route {KE_MEAN:.4e} V s/rad, sd {KE_SD / KE_MEAN * 100:.2f} % over {len(ok)} cycles = {KE_MEAN * 2 * np.pi * 1e3:.3f} mV per rev/s "
      f"= {KE_MEAN * 2 * np.pi / N_R * 1e3:.4f} mV per ripple Hz")
for v in COLOR:
    g = ok[ok["level"] == v]["ke_coast"]
    print(f"  {v:g} V: {g.mean():.4e} V s/rad, sd {g.std() / g.mean() * 100:.2f} %, n {len(g)}, {(g.mean() / KE_MEAN - 1) * 100:+.2f} % from the pooled mean")
for d in (1, -1):
    g = ok[ok["dir"] == d]["ke_coast"]
    print(f"  direction {d:+d}: {g.mean():.4e} V s/rad, sd {g.std() / g.mean() * 100:.2f} %, n {len(g)}, {(g.mean() / KE_MEAN - 1) * 100:+.2f} % from the pooled mean")

Each route's mean, spread and standard error over the turned cycles, then the coast route per level and duty, and the pooled value in the other units, per revolution per second and per ripple hertz. A level or a direction that sits off the pooled mean by more than the spread is a finding, and a duty whose three repeats scatter wider than the rest says the count was unsure there.

### The paired difference

The spin route over the coast route, less one, cycle by cycle against the coast's speed. Below it the difference is split into its two parts, the spin's counted speed over the coast's and $E_0$ over the window's mean EMF. The thing to see is whether the spin route reads high or low on the whole, and which part carries the difference.

In [ ]:
pair = ok.dropna(subset=["ke_spin"]).copy()
pair["diff_pct"] = (pair["ke_spin"] / pair["ke_coast"] - 1) * 100
pair["speed_part_pct"] = (pair["f_coast"] / pair["f_spin"] - 1) * 100       # the speed in the denominator: the coast's over the spin's
pair["emf_part_pct"] = (pair["e0_v"] / pair["emf_v"] - 1) * 100             # the EMF in the numerator: E0 over the window's mean
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, c, title in zip(ax, ["diff_pct", "speed_part_pct", "emf_part_pct"],
                       ["spin route over coast route", "speed: the coast's count over the spin's", "EMF: $E_0$ at the cut over the window's mean"]):
    for v in COLOR:
        for d in (1, -1):
            h = pair[(pair["level"] == v) & (pair["dir"] == d)]
            a.plot(h["speed_rev_s"], h[c], MARK[d], color=COLOR[v], mfc=COLOR[v] if d == 1 else "none", label=f"{v:g} V, direction {d:+d}")
    a.axhline(0, color=INK, lw=0.5); a.axhline(pair[c].mean(), color=INK, lw=0.8, ls=":"); a.set_title(title); a.set_xlabel("speed by the coast's count (rev/s)"); a.grid(alpha=0.3)
ax[0].set_ylabel("less one (%)"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
DIFF_MEAN, DIFF_SE = pair["diff_pct"].mean(), pair["diff_pct"].std() / np.sqrt(len(pair))
print(f"spin / coast - 1: {DIFF_MEAN:+.2f} % (sd {pair['diff_pct'].std():.2f} %, se {DIFF_SE:.2f} %, n {len(pair)})")
print(f"  the speed part {pair['speed_part_pct'].mean():+.2f} % (sd {pair['speed_part_pct'].std():.2f}), the EMF part {pair['emf_part_pct'].mean():+.2f} % (sd {pair['emf_part_pct'].std():.2f})")

The two routes share the ripple count's method and differ in where the EMF and the speed come from, so a paired difference near zero with a spread of a percent or so says the spin's speed carried across the gap and the extrapolation to the cut lands on the coast's own reading. A difference that grows with speed points at the extrapolation, and one that is flat and offset points at the speed.

### The EMF against the speed

Every turned cycle's window EMF against the speed counted in the same rows, with the line through the origin at the cycles' mean $K_e$, and under it each cycle's $K_e$ from that mean. The thing to see is whether the points lie on one line through the origin from the slowest cycle to the fastest, and whether the two levels and the two directions sit on the same line.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
x = np.linspace(0, ok["w_coast"].max() * 1.05, 10)
for v in COLOR:
    for d in (1, -1):
        h = ok[(ok["level"] == v) & (ok["dir"] == d)]
        kw = dict(ls="none", marker=MARK[d], ms=6, color=COLOR[v], mfc=COLOR[v] if d == 1 else "none", label=f"{v:g} V, direction {d:+d}")
        ax[0].plot(h["w_coast"], h["emf_v"], **kw); ax[1].plot(h["w_coast"], (h["ke_coast"] / KE_MEAN - 1) * 100, **kw)
ax[0].plot(x, KE_MEAN * x, "-", color=INK, lw=1, label=f"$K_e$ = {KE_MEAN * 1e3:.4f} mV s/rad, the cycles' mean, through the origin")
ax[0].set_ylabel("EMF over the counted window (V)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3); ax[0].set_title("the EMF against the speed counted in the same rows")
ax[1].axhline(0, color=INK, lw=0.5); ax[1].set_ylabel("per-cycle $K_e$ from the mean (%)"); ax[1].set_xlabel("speed (rad/s)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
lin = np.polyfit(ok["w_coast"], ok["emf_v"], 1)
print(f"a free line through the cycles: slope {lin[0] * 1e3:.4f} mV s/rad, intercept {lin[1] * 1e3:+.1f} mV; the line through the origin: {KE_MEAN * 1e3:.4f} mV s/rad")
print(f"EMF {ok['emf_v'].min():.2f} to {ok['emf_v'].max():.2f} V over {ok['speed_rev_s'].min():.0f} to {ok['speed_rev_s'].max():.0f} rev/s, "
      f"ripple {ok['f_coast'].min():.0f} to {ok['f_coast'].max():.0f} Hz, {FS / ok['f_coast'].max():.1f} rows per ripple cycle at the top")

A free line with an intercept is printed beside the line through the origin. An intercept of more than a few millivolts, which is about what a count of tap difference is worth, says the EMF is not proportional to the speed over this range, or that the rest difference is not the right zero.

### Ke down the coast

The same 10 ms window stepped down each coast, one cycle per level and duty, so $K_e$ is read at every speed the rotor passes through on its way to a stop. The thing to see is whether the read stays flat as the speed falls, and where the count gives out.

In [ ]:
WIN = COAST_TO - COAST_FROM     # 200 rows, 10 ms, stepped down the coast
rows = []
for _, C in cycles[cycles["repeat"] == 0].iterrows():       # one cycle per level and duty is enough to see the shape
    df, meta = tel.load(cycle_folder(C["k"]), BOARD)
    rest = tel.baseline(df); zero_e = (rest["vmotor_a"] - rest["vmotor_b"]).median()
    for d, sp, co in pairs(df):
        z = cut_row(co)
        if z is None:
            continue
        e = (((co["vmotor_a"] - co["vmotor_b"]) * d).to_numpy(float)[z:] - zero_e) * V_PER_COUNT
        for start in range(COAST_FROM, len(e) - WIN, WIN):
            seg = e[start:start + WIN]
            if seg.mean() < TURN_V:
                break
            f0, prom = line_freq(seg)
            f, b0, b1, n = count_cycles(seg, f0)
            if n < 2:
                break
            w = 2 * np.pi * f / N_R
            rows.append({"k": int(C["k"]), "level": float(C["volts_set"]), "duty": int(C["duty_pct"]), "dir": d, "t_s": (start + WIN / 2) / FS,
                         "w": w, "emf_v": seg[b0:b1].mean(), "ke": seg[b0:b1].mean() / w, "cycles": n})
ALONG = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 4.5))
for v in COLOR:
    for duty, g in ALONG[ALONG["level"] == v].groupby("duty"):
        ax.plot(g["w"] / (2 * np.pi), (g["ke"] / KE_MEAN - 1) * 100, ".", ms=4, color=SHADE[v][duty], label=f"{v:g} V, {duty} %")
ax.axhline(0, color=INK, lw=0.5); ax.set_xlabel("speed in the window (rev/s)"); ax.set_ylabel("$K_e$ in the window, from the cycles' mean (%)")
ax.set_title("$K_e$ read in 10 ms windows down each coast, one cycle per level and duty"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
dev = (ALONG["ke"] / KE_MEAN - 1) * 100
print(f"{len(ALONG)} windows, counted down to {ALONG['w'].min() / (2 * np.pi):.0f} rev/s; the read sits {dev.quantile(0.05):+.2f} to {dev.quantile(0.95):+.2f} % from the mean "
      f"at the 5th and 95th percentile, median {dev.median():+.2f} %")

Each coast traces its own path down in speed, and the paths from different duties overlap where their speeds do, so a shape shared by every path is the motor's and a shape that belongs to one path is that window's. A read that climbs or falls at the low end says either that the EMF is not proportional there or that the count slips when a window holds only a few ripple cycles, and the number of cycles per window is in the table to tell those apart.

### The two taps in the coast

A floating winding splits its EMF between the two terminals around the bias node, the positive one rising by about half the EMF and the negative one falling by the same, and the difference reads the whole. The positive and the negative terminal's excursion from its own rest against the window EMF, both as magnitudes, with the half line. The thing to see is where the two leave the half line, which would mean one terminal is held by something in the driver, since the difference is only as good as both taps staying in range.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(0, ok["emf_v"].max() * 1.05, 10)
for v in COLOR:
    g = ok[ok["level"] == v]
    ax.plot(g["emf_v"], g["tap_pos_v"], "o", color=COLOR[v], label=f"{v:g} V, the positive terminal")
    ax.plot(g["emf_v"], -g["tap_neg_v"], "^", color=COLOR[v], mfc="none", label=f"{v:g} V, the negative terminal, sign flipped")
ax.plot(x, x / 2, "-", color=INK, lw=0.8, label="half the EMF")
ax.axhline(BOARD.vb_nom_v, color=MUTED, lw=0.8, ls="--", label=f"the bias node's nominal {BOARD.vb_nom_v:.2f} V, where the negative terminal would reach ground")
ax.set_xlabel("EMF over the counted window, tap A - tap B (V)"); ax.set_ylabel("one terminal's excursion from its rest (V)")
ax.set_title("how the EMF splits across the two terminals"); ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
split = (ok["tap_pos_v"] + ok["tap_neg_v"]) / ok["emf_v"] * 100     # the asymmetry: positive excursion less negative magnitude, as a share of the EMF
print(f"positive less negative excursion, as a share of the EMF: {split.mean():+.1f} % on average, {split.min():+.1f} to {split.max():+.1f} %; "
      f"the negative terminal reaches {ok['tap_neg_v'].min():+.2f} V from its rest at the most")

Two terminals on the half line up to the highest EMF say both taps stayed inside their range and the difference carries the whole EMF. A negative terminal that stops falling while the positive one keeps rising says a clamp in the driver has taken it, and the difference above that point rests on the positive tap alone.

### The results file

$K_e$ by the coast route over the turned cycles, the cycles' spread as the bounds, in volt seconds per radian and in millivolts per revolution per second, with the per-level and per-direction values, the paired difference of the spin route with its standard error, the EMF line's intercept, the speed and EMF ranges, the settle figures, and the per-cycle table for the no-load-current experiment, one row per spin and coast pair with the speed, the EMF, the spin's current, the EMF at the cut and its slope.

In [ ]:
q = results.quantity

def native(v):
    # JSON takes Python scalars only, and a NaN becomes null
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return None if not np.isfinite(v) else float(v)
    return v

def spread(x, unit):
    x = x.dropna()
    return q(x.mean(), x.mean() - x.std(), x.mean() + x.std(), unit)

CYCLE_COLS = {"k": "k", "level": "level", "duty": "duty", "dir": "direction", "turned": "turned", "speed_rev_s": "speed_rev_s", "emf_v": "emf_v",
              "i_spin_a": "i_spin_a", "ke_coast": "ke", "ke_spin": "ke_spin", "e0_v": "e0_v", "dedt_v_per_s": "dedt_v_per_s", "f_coast": "f_coast_hz",
              "coast_cycles": "coast_cycles", "rail_v": "rail_v", "duty_applied": "duty_applied_pct"}
cycle_list = [{name: native(r.get(c, np.nan)) for c, name in CYCLE_COLS.items()} for r in CYC.to_dict("records")]
results.write("2-motor", "bemf-coast", CAPTURE, pair=MOTOR.key,
              notes="cycles of a 1.5 s spin and a 0.8 s coast with the bridge open, both directions, 15 to 60 % at 5.0 and 7.4 V; ke_v_s_per_rad is the coast "
                    "route, the EMF over a whole number of ripple cycles 1 to 11 ms past the cut over the speed counted in the same rows, 6 ripple events per "
                    "revolution, mean over the turned cycles with the cycles' sd as the bounds; the spin route is E0 at the cut over the spin's current-ripple "
                    "speed and is reported as its paired difference only; terminal volts through the stage 1 terminal-chain scale with each cycle's rest "
                    "difference as the zero, current through current-scale with the in-drive offset taken out; cycles is the per-cycle table no-load-current reads",
              ke_v_s_per_rad=q(KE_MEAN, KE_MEAN - KE_SD, KE_MEAN + KE_SD, "V s/rad"),
              ke_mv_per_rev_s=q(KE_MEAN * 2 * np.pi * 1e3, (KE_MEAN - KE_SD) * 2 * np.pi * 1e3, (KE_MEAN + KE_SD) * 2 * np.pi * 1e3, "mV per rev/s"),
              ke_mv_per_ripple_hz=q(KE_MEAN * 2 * np.pi / N_R * 1e3, unit="mV/Hz"),
              ke_sd_pct=q(KE_SD / KE_MEAN * 100, unit="%"),
              ke_se_pct=q(KE_SD / np.sqrt(len(ok)) / KE_MEAN * 100, unit="%"),
              ke_by_level={f"{v:g} V": spread(ok[ok["level"] == v]["ke_coast"], "V s/rad") for v in COLOR},
              ke_by_direction={f"{d:+d}": spread(ok[ok["dir"] == d]["ke_coast"], "V s/rad") for d in (1, -1)},
              ke_spin_route_pct_diff=q(DIFF_MEAN, DIFF_MEAN - DIFF_SE, DIFF_MEAN + DIFF_SE, "%"),
              emf_line_intercept_mv=q(lin[1] * 1e3, unit="mV"),
              emf_line_slope_v_s_per_rad=q(lin[0], unit="V s/rad"),
              speed_range_rev_s=q(ok["speed_rev_s"].max(), lo=ok["speed_rev_s"].min(), hi=ok["speed_rev_s"].max(), unit="rev/s"),
              emf_range_v=q(ok["emf_v"].max(), lo=ok["emf_v"].min(), hi=ok["emf_v"].max(), unit="V"),
              ripple_per_rev=q(N_R, unit="events/rev"),
              cycles_turned=q(len(ok), unit="count"),
              cycles_total=q(len(CYC), unit="count"),
              spin_tail_tau_ms=q(TAIL["tail_tau_ms"].median(), TAIL["tail_tau_ms"].min(), TAIL["tail_tau_ms"].max(), "ms"),
              spin_window_over_asymptote_ma=q(TAIL["window_less_inf_ma"].mean(), TAIL["window_less_inf_ma"].min(), TAIL["window_less_inf_ma"].max(), "mA"),
              spin_speed_drift_pct=q(TAIL["speed_drift_pct"].mean(), TAIL["speed_drift_pct"].min(), TAIL["speed_drift_pct"].max(), "%"),
              tap_split_asymmetry_pct=q(split.mean(), split.min(), split.max(), "%"),
              cycles=cycle_list)

## Limitations

- The speed has no outside ruler. The count rests on the ripple being six events per revolution, a parts count, 3 slots times 2 brushes, and on the band-pass staying on that line. A camera on the horn grades the ripple count in stage 3, through the gear train, and nothing here does on the bare shaft.
- The terminal scale is the stage 1 result through the meter, 0.1 % at best, and the zero is each cycle's own rest difference. The terminal-chain result also carries an offset of about 2.4 counts read under DC current, which a coast does not have and this notebook does not apply. At the lowest EMF, a few tenths of a volt, one count of difference is a percent or two, so the slow cycles carry the chain's floor and the fast ones do not.
- The EMF decays through the 10 ms window, so the window's mean EMF and its mean frequency are both averages over a changing speed. The ratio is what is read. The windows down the coast show whether the ratio moves with the speed, and not whether the average itself is biased by the decay inside a window.
- The current in the spin window is the crest sample under PWM at 15 to 60 % duty, and the current-scale grade is DC at 100 %. A short drive window is known to read differently from a long one, so the spin's current is carried as read, with the offset taken out, for the no-load-current experiment to weigh.
- The motor's temperature is not read. The cycles are short and rest twenty seconds, so the copper stays near the can, but the magnets' flux moves with temperature too and a warm motor would read a lower $K_e$. Nothing here covers that.
- Above about 2 V of EMF the negative terminal would want to sit below ground, since the split runs around a bias node of 0.62 V, and whatever in the driver holds it there is not characterized. The two-taps figure is the check this capture allows.
- The DPS is the supply, trimmed to the level by the board's own rail. The rail sags under the spin by the current times the leads' and the diode's drop, which the rail column records and nothing here needs.
- One motor. A second motor of the same batch is read with the meter in rest-resistance for the resistance alone, and nothing says how far $K_e$ moves between cans.

## What earlier work established

Before this stage the same route was tried on a motor of this family out of its servo with a 300 ms spin, aimed at understanding the method rather than at grading a motor. Its data is not reused here, but its conclusions set this capture up.

- The coast route read $K_e$ with a spread of a few tenths of a percent over a dozen cycles and sat flat across 20 to 40 % duty on a 5 V rail, and the spin route agreed with it paired cycle by cycle to within a percent with a wider spread. The coast route was the one to keep, since it needs no current and no resistance.
- A 300 ms spin was still accelerating in its window. At 40 % the current fell a few milliamps across the last 100 ms while the speed climbed a percent, the tails fitted time constants of 70 to 170 ms, and the no-load currents read a few milliamps high for it. That is why the spin here is 1.5 s.
- A coast with the bridge open reads the EMF, and a brake segment with the low side shorted reads about 4 % of it, a resistive residue and not the EMF. The first floating samples after the cut read low while the taps settle, so the EMF is read over whole ripple periods past the first millisecond and not from the first sample.
- Under drive the current ripple can carry a stronger line at three events per revolution than at six above about half duty, and a free peak search then halves the speed. The spin's count here is seeded by the coast's line for that reason.
- A single terminal reads about half the EMF, the two taps splitting it around the bias node, so the difference is the EMF and one tap on its own is not.
- An in-servo value of the same constant from a drive fit, with the gear train on and the resistance pinned, sat several percent from the bare coast's, and nothing separated the board, the route and the gear train. Stage 3 repeats this experiment in situ and is graded against this notebook for that reason.

## Open questions

- Whether one terminal is held by the driver at the highest EMF. The two-taps figure says where the split leaves the half line, if it does. A scope on the negative terminal through a 60 % coast at 7.4 V reads whether it sits below ground and by how much, and the same cycles with the rail at 5.0 V, where the EMF stays lower, read the same duties without it.
- Whether the speed moves between the spin window and the cut. The rows between the two streams are not recorded. The paired difference of the two routes bounds the move, and the spin's own drift across the window, from the settle table, says how much of the difference the window itself allows. A paired difference larger than that drift would say the gap carried a speed change, and a capture that reads the spin's last rows and the coast's first rows in one stream would settle it, which the sweep tool does not do today.
- Whether $K_e$ holds at the low end, below the slowest cycle here. The windows down the coast read it to where the count gives out, and a capture with 10 and 12 % duties at 5.0 V spins the rotor at those speeds from the start, where a coast has whole cycles to count in its first 10 ms and the window is not on the tail of a decay.
- Whether $K_e$ moves with the motor's temperature. The same cycles run right after a 30 s spin, with the thermocouple on the can, read the constant warm against cold, and the magnets' coefficient would show as a lower EMF at the same counted speed.
- Whether the count's floor is the chain. The slow cycles sit a count or two of tap difference above their zero, and the rest-noise floor says what one row carries. A capture with the coast's rows averaged on the board before they are sent, or a longer window at the slow duties, says whether the slow cycles' scatter is the chain's noise or the count.
- Whether the ripple is six events per revolution on this motor. The count assumes it. A camera on a mark on the pinion, or the rest-resistance pass's twelve holes read against the ripple's phase, pins the events per revolution with no assumption.

## Conclusion

To be written from the capture.